# AI Systems Validation Lab


## Purpose

This lab introduces artificial intelligence as a validation problem, not merely a modeling problem.

You will build a synthetic classification system, train a baseline model, evaluate performance, and produce an audit-ready metrics table.

Learning goals:

- Understand the relationship between data, model, prediction, and evaluation.
- Compute accuracy, precision, recall, F1, and ROC AUC.
- Separate model capability from system trustworthiness.
- Prepare outputs that could be used in an AI governance workflow.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42

## Synthetic AI System Dataset

This dataset is synthetic. It represents the structure of an AI decision-support problem without using private, regulated, or sensitive data.

In [ ]:
X, y = make_classification(
    n_samples=6000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.65, 0.35],
    class_sep=0.9,
    random_state=RANDOM_SEED,
)

frame = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
frame["target"] = y

rng = np.random.default_rng(RANDOM_SEED)
frame["group"] = rng.choice(["A", "B", "C"], size=len(frame), p=[0.50, 0.30, 0.20])

frame.head()

## Train a Baseline Model

A baseline model is not the final system. It is a reference point for disciplined comparison.

In [ ]:
features = [c for c in frame.columns if c.startswith("feature_")]

X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    frame[features],
    frame["target"],
    frame["group"],
    test_size=0.30,
    stratify=frame["target"],
    random_state=RANDOM_SEED,
)

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]
)

model.fit(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]
predictions = (scores >= 0.50).astype(int)

## Model Evaluation

Aggregate metrics are necessary but incomplete. They summarize performance but do not reveal who bears different kinds of error.

In [ ]:
metrics = {
    "accuracy": accuracy_score(y_test, predictions),
    "precision": precision_score(y_test, predictions, zero_division=0),
    "recall": recall_score(y_test, predictions, zero_division=0),
    "f1": f1_score(y_test, predictions, zero_division=0),
    "roc_auc": roc_auc_score(y_test, scores),
}

pd.DataFrame([metrics])

In [ ]:
print(classification_report(y_test, predictions, zero_division=0))
print(confusion_matrix(y_test, predictions))

## Interpretation Checklist

Use these prompts before treating model performance as trustworthy:

1. What does a false positive mean in this domain?
2. What does a false negative mean?
3. Are the data representative of deployment conditions?
4. Are thresholds aligned with the real decision context?
5. Does performance vary across groups, time periods, or environments?